[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_11/notebook_11_2_federated_learning.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 11.2: Federated Learning for Healthcare

**Volume 1: Foundations of AI in Healthcare**  
**Chapter 11: Privacy, Security, and Trustworthy AI**

---

## Introduction

Healthcare data is naturally distributed across multiple institutions—hospitals, clinics, research centers—each holding valuable patient records that could improve AI models if combined. However, sharing raw patient data faces significant barriers:

- **Privacy regulations**: HIPAA, GDPR, and local laws restrict data sharing
- **Institutional policies**: Hospitals are reluctant to share competitive advantages
- **Technical challenges**: Data transfer is expensive and time-consuming
- **Patient trust**: Concerns about data misuse and breaches

**Federated Learning (FL)** offers an elegant solution: train models collaboratively **without sharing raw data**. Instead of centralizing data, FL brings the model to the data:

1. Each institution trains a local model on its own data
2. Only model updates (gradients/weights) are shared with a central server
3. The server aggregates updates into a global model
4. The improved global model is sent back to institutions
5. Process repeats for multiple rounds

### Clinical Motivation: Rare Disease Prediction

Consider a rare genetic disorder affecting 1 in 50,000 people. Individual hospitals may have only 5-10 cases, insufficient for training robust models. Through federated learning, 10 hospitals can collaboratively train a model on 50-100 cases **without sharing patient data**, dramatically improving diagnostic accuracy.

### Learning Objectives

By the end of this notebook, you will:

1. Understand the **FedAvg** (Federated Averaging) algorithm
2. Implement federated learning for multi-hospital collaboration
3. Handle **non-IID** (non-Independent and Identically Distributed) data across institutions
4. Compare federated learning with centralized and local-only training
5. Understand privacy guarantees and limitations of FL
6. Explore secure aggregation protocols

---

## Setup

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

## Part 1: Federated Averaging (FedAvg) Algorithm

### Mathematical Foundation

The **FedAvg** algorithm (McMahan et al., 2017) is the foundational protocol for federated learning:

**Goal**: Minimize the global loss function:

$$
\min_{w} F(w) = \sum_{k=1}^{K} \frac{n_k}{n} F_k(w)
$$

Where:
- $K$ = number of participating institutions (hospitals)
- $n_k$ = number of data points at institution $k$
- $n = \sum_{k=1}^{K} n_k$ = total data points across all institutions
- $F_k(w)$ = local loss function at institution $k$
- $w$ = model parameters (weights)

### FedAvg Protocol

**Server side** (each global round $t$):
1. Initialize global model $w^{(t)}$
2. Sample a subset of institutions $S_t \subseteq \{1, ..., K\}$
3. Send $w^{(t)}$ to each selected institution
4. Receive updated models $w_k^{(t+1)}$ from each institution
5. Aggregate updates:

$$
w^{(t+1)} = \sum_{k \in S_t} \frac{n_k}{\sum_{j \in S_t} n_j} w_k^{(t+1)}
$$

**Client side** (institution $k$ in round $t$):
1. Receive global model $w^{(t)}$
2. Train locally for $E$ epochs on local data:

$$
w_k^{(t+1)} = w^{(t)} - \eta \nabla F_k(w^{(t)})
$$

3. Send updated model $w_k^{(t+1)}$ to server

### Key Properties

- **Privacy-preserving**: Raw data never leaves institutions
- **Communication-efficient**: Only model updates are transmitted
- **Weighted aggregation**: Larger datasets have proportionally more influence
- **Convergence guarantee**: Under certain conditions, FedAvg converges to a good solution

In [ ]:
class FederatedLearning:
    """
    Federated Learning implementation using FedAvg algorithm.

    This class simulates a federated learning environment where multiple
    institutions (hospitals) collaborate to train a shared model without
    sharing their raw patient data.
    """

    def __init__(self, n_institutions: int, input_dim: int, learning_rate: float = 0.01):
        """
        Initialize federated learning system.

        Args:
            n_institutions: Number of participating hospitals
            input_dim: Number of input features
            learning_rate: Learning rate for local training
        """
        self.n_institutions = n_institutions
        self.input_dim = input_dim
        self.learning_rate = learning_rate

        # Initialize global model (logistic regression)
        self.global_weights = np.zeros(input_dim)
        self.global_bias = 0.0

        # Track metrics
        self.history = {
            'global_loss': [],
            'global_accuracy': [],
            'local_losses': [[] for _ in range(n_institutions)],
            'local_accuracies': [[] for _ in range(n_institutions)]
        }

    def sigmoid(self, z):
        """Sigmoid activation function."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def compute_loss(self, X, y, weights, bias):
        """
        Compute logistic loss (binary cross-entropy).

        L(w, b) = -1/n * Σ[y_i * log(ŷ_i) + (1-y_i) * log(1-ŷ_i)]
        """
        n = len(y)
        z = np.dot(X, weights) + bias
        predictions = self.sigmoid(z)

        # Prevent log(0)
        predictions = np.clip(predictions, 1e-10, 1 - 1e-10)

        loss = -np.mean(y * np.log(predictions) + (1 - y) * np.log(1 - predictions))
        return loss

    def compute_gradients(self, X, y, weights, bias):
        """
        Compute gradients for logistic regression.

        ∇w = 1/n * X^T * (ŷ - y)
        ∇b = 1/n * Σ(ŷ - y)
        """
        n = len(y)
        z = np.dot(X, weights) + bias
        predictions = self.sigmoid(z)

        errors = predictions - y
        grad_w = np.dot(X.T, errors) / n
        grad_b = np.mean(errors)

        return grad_w, grad_b

    def local_train(self, X, y, initial_weights, initial_bias, epochs=5):
        """
        Train model locally at one institution.

        Args:
            X: Local training features
            y: Local training labels
            initial_weights: Starting weights from global model
            initial_bias: Starting bias from global model
            epochs: Number of local training epochs

        Returns:
            Updated weights, bias, and metrics
        """
        weights = initial_weights.copy()
        bias = initial_bias

        for epoch in range(epochs):
            # Compute gradients
            grad_w, grad_b = self.compute_gradients(X, y, weights, bias)

            # Update parameters
            weights -= self.learning_rate * grad_w
            bias -= self.learning_rate * grad_b

        # Compute final loss and accuracy
        loss = self.compute_loss(X, y, weights, bias)
        predictions = (self.sigmoid(np.dot(X, weights) + bias) >= 0.5).astype(int)
        accuracy = np.mean(predictions == y)

        return weights, bias, loss, accuracy

    def aggregate_models(self, local_updates: List[Tuple], data_sizes: List[int]):
        """
        Aggregate local model updates using weighted averaging.

        FedAvg aggregation:
        w_global = Σ (n_k / n_total) * w_k

        Args:
            local_updates: List of (weights, bias) tuples from each institution
            data_sizes: Number of training samples at each institution

        Returns:
            Aggregated global weights and bias
        """
        total_samples = sum(data_sizes)

        # Weighted averaging
        aggregated_weights = np.zeros_like(self.global_weights)
        aggregated_bias = 0.0

        for (weights, bias), n_samples in zip(local_updates, data_sizes):
            weight_factor = n_samples / total_samples
            aggregated_weights += weight_factor * weights
            aggregated_bias += weight_factor * bias

        return aggregated_weights, aggregated_bias

    def train_federated(self, institutions_data: List[Tuple],
                       global_test_X, global_test_y,
                       n_rounds: int = 20, local_epochs: int = 5):
        """
        Execute federated learning for multiple rounds.

        Args:
            institutions_data: List of (X_train, y_train) for each institution
            global_test_X: Global test features for evaluation
            global_test_y: Global test labels for evaluation
            n_rounds: Number of federated learning rounds
            local_epochs: Epochs of local training per round
        """
        print(f"Starting Federated Learning with {self.n_institutions} institutions...")
        print(f"Rounds: {n_rounds}, Local epochs per round: {local_epochs}\n")

        for round_num in range(n_rounds):
            print(f"Round {round_num + 1}/{n_rounds}")

            # Step 1: Each institution trains locally
            local_updates = []
            data_sizes = []

            for inst_id, (X_train, y_train) in enumerate(institutions_data):
                # Local training
                weights, bias, loss, accuracy = self.local_train(
                    X_train, y_train,
                    self.global_weights, self.global_bias,
                    epochs=local_epochs
                )

                local_updates.append((weights, bias))
                data_sizes.append(len(y_train))

                # Track local metrics
                self.history['local_losses'][inst_id].append(loss)
                self.history['local_accuracies'][inst_id].append(accuracy)

                print(f"  Institution {inst_id + 1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}")

            # Step 2: Server aggregates updates
            self.global_weights, self.global_bias = self.aggregate_models(
                local_updates, data_sizes
            )

            # Step 3: Evaluate global model
            global_loss = self.compute_loss(
                global_test_X, global_test_y,
                self.global_weights, self.global_bias
            )
            global_preds = (self.sigmoid(
                np.dot(global_test_X, self.global_weights) + self.global_bias
            ) >= 0.5).astype(int)
            global_accuracy = np.mean(global_preds == global_test_y)

            self.history['global_loss'].append(global_loss)
            self.history['global_accuracy'].append(global_accuracy)

            print(f"  Global Model: Loss={global_loss:.4f}, Accuracy={global_accuracy:.4f}\n")

        print("Federated Learning completed!")

    def predict(self, X):
        """Make predictions using the global model."""
        z = np.dot(X, self.global_weights) + self.global_bias
        return (self.sigmoid(z) >= 0.5).astype(int)

    def predict_proba(self, X):
        """Return prediction probabilities."""
        z = np.dot(X, self.global_weights) + self.global_bias
        return self.sigmoid(z)

print("FederatedLearning class defined successfully!")

---

## Part 2: Simulating Multi-Hospital Collaboration

### Scenario: Rare Genetic Disorder Prediction

We simulate 10 hospitals collaborating to predict a rare genetic disorder. Each hospital has:

- **Different patient populations** (non-IID data)
- **Varying dataset sizes** (50-200 patients each)
- **Local privacy constraints** (cannot share raw data)

**Features** (simulated clinical measurements):
- Genetic markers (10 features)
- Clinical biomarkers (5 features)
- Demographic factors (3 features)

**Target**: Binary diagnosis (0 = No disorder, 1 = Disorder present)

In [ ]:
def generate_hospital_data(n_hospitals=10, base_samples=100, feature_dim=18):
    """
    Generate synthetic medical data for multiple hospitals.

    Simulates non-IID data distribution:
    - Each hospital has different patient demographics
    - Varying prevalence rates
    - Different dataset sizes

    Args:
        n_hospitals: Number of participating hospitals
        base_samples: Base number of samples per hospital
        feature_dim: Number of features

    Returns:
        institutions_data: List of (X, y) for each hospital
        global_test_data: Combined test set
        feature_names: Names of features
    """
    np.random.seed(42)

    institutions_data = []
    all_test_X = []
    all_test_y = []

    # Feature names
    feature_names = (
        [f'genetic_marker_{i+1}' for i in range(10)] +
        [f'biomarker_{i+1}' for i in range(5)] +
        ['age', 'bmi', 'family_history']
    )

    print("Generating data for hospitals...\n")

    for hospital_id in range(n_hospitals):
        # Varying sample sizes
        n_samples = base_samples + np.random.randint(-30, 50)

        # Create non-IID distribution by shifting feature means
        # Each hospital serves slightly different patient populations
        hospital_shift = np.random.randn(feature_dim) * 0.3

        # Generate features
        X = np.random.randn(n_samples, feature_dim) + hospital_shift

        # Generate labels with hospital-specific prevalence
        # True relationship: disorder depends on genetic markers + biomarkers
        true_weights = np.concatenate([
            np.array([0.5, 0.3, -0.4, 0.6, 0.2, -0.3, 0.4, 0.1, -0.2, 0.3]),  # genetic
            np.array([0.4, -0.3, 0.5, 0.2, -0.4]),  # biomarkers
            np.array([0.1, 0.0, 0.6])  # demographics
        ])

        # Compute probabilities
        logits = np.dot(X, true_weights) + np.random.randn(n_samples) * 0.5
        probabilities = 1 / (1 + np.exp(-logits))

        # Adjust prevalence rate per hospital (5-15%)
        hospital_prevalence = 0.10 + np.random.uniform(-0.05, 0.05)
        threshold = np.percentile(probabilities, (1 - hospital_prevalence) * 100)
        y = (probabilities >= threshold).astype(int)

        # Split into train/test (80/20 for each hospital)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=hospital_id, stratify=y
        )

        # Standardize features
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        institutions_data.append((X_train, y_train))
        all_test_X.append(X_test)
        all_test_y.append(y_test)

        print(f"Hospital {hospital_id + 1}:")
        print(f"  Training samples: {len(y_train)} (Positive: {y_train.sum()}, {y_train.mean():.2%})")
        print(f"  Test samples: {len(y_test)} (Positive: {y_test.sum()}, {y_test.mean():.2%})")

    # Combine all test sets for global evaluation
    global_test_X = np.vstack(all_test_X)
    global_test_y = np.concatenate(all_test_y)

    print(f"\nGlobal test set: {len(global_test_y)} samples")
    print(f"Overall prevalence: {global_test_y.mean():.2%}")

    return institutions_data, (global_test_X, global_test_y), feature_names

# Generate data
institutions_data, (global_test_X, global_test_y), feature_names = generate_hospital_data(
    n_hospitals=10,
    base_samples=100,
    feature_dim=18
)

### Visualizing Data Distribution Across Hospitals

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dataset sizes
train_sizes = [len(y) for X, y in institutions_data]
axes[0].bar(range(1, 11), train_sizes, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Hospital ID', fontsize=12)
axes[0].set_ylabel('Number of Training Samples', fontsize=12)
axes[0].set_title('Training Data Distribution Across Hospitals', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, 11))
axes[0].grid(axis='y', alpha=0.3)

# Prevalence rates
prevalence_rates = [y.mean() for X, y in institutions_data]
axes[1].bar(range(1, 11), prevalence_rates, color='coral', alpha=0.7)
axes[1].axhline(y=np.mean(prevalence_rates), color='red', linestyle='--',
                label=f'Mean: {np.mean(prevalence_rates):.2%}')
axes[1].set_xlabel('Hospital ID', fontsize=12)
axes[1].set_ylabel('Disease Prevalence', fontsize=12)
axes[1].set_title('Non-IID Data: Varying Prevalence Rates', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(1, 11))
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Data Characteristics:")
print(f"Total training samples: {sum(train_sizes)}")
print(f"Average samples per hospital: {np.mean(train_sizes):.1f} ± {np.std(train_sizes):.1f}")
print(f"Prevalence range: {min(prevalence_rates):.2%} - {max(prevalence_rates):.2%}")
print("\n⚠️ Non-IID Challenge: Different hospitals have different data distributions!")

---

## Part 3: Training Federated Model

Now we train a federated model across all 10 hospitals using the FedAvg algorithm.

In [ ]:
# Initialize federated learning system
fl_system = FederatedLearning(
    n_institutions=10,
    input_dim=18,
    learning_rate=0.1
)

# Train federated model
fl_system.train_federated(
    institutions_data=institutions_data,
    global_test_X=global_test_X,
    global_test_y=global_test_y,
    n_rounds=15,
    local_epochs=5
)

### Visualizing Federated Learning Progress

In [ ]:
# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Global model performance
rounds = range(1, len(fl_system.history['global_loss']) + 1)
axes[0].plot(rounds, fl_system.history['global_loss'], marker='o', linewidth=2,
            markersize=6, color='darkblue', label='Global Loss')
axes[0].set_xlabel('Federated Round', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Global Model: Training Loss', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(rounds, fl_system.history['global_accuracy'], marker='o', linewidth=2,
            markersize=6, color='darkgreen', label='Global Accuracy')
axes[1].set_xlabel('Federated Round', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Global Model: Test Accuracy', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Final performance
final_accuracy = fl_system.history['global_accuracy'][-1]
final_loss = fl_system.history['global_loss'][-1]

print("\n🎯 Final Federated Model Performance:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Loss: {final_loss:.4f}")

---

## Part 4: Comparison with Baseline Approaches

To demonstrate the value of federated learning, we compare three approaches:

1. **Local-only training**: Each hospital trains independently on its own data
2. **Centralized training**: All data pooled together (ideal but privacy-violating)
3. **Federated learning**: Our FedAvg approach (privacy-preserving collaboration)

This comparison shows:
- **Local-only** suffers from small dataset sizes
- **Centralized** achieves best performance but violates privacy
- **Federated** approaches centralized performance while preserving privacy

In [ ]:
# 1. Local-only training (each hospital trains independently)
print("Training local-only models...\n")
local_accuracies = []

for hospital_id, (X_train, y_train) in enumerate(institutions_data):
    # Train sklearn model locally
    local_model = LogisticRegression(max_iter=1000, random_state=42)
    local_model.fit(X_train, y_train)

    # Evaluate on global test set
    accuracy = local_model.score(global_test_X, global_test_y)
    local_accuracies.append(accuracy)
    print(f"Hospital {hospital_id + 1} - Local model accuracy: {accuracy:.4f}")

avg_local_accuracy = np.mean(local_accuracies)
print(f"\nAverage local-only accuracy: {avg_local_accuracy:.4f} ± {np.std(local_accuracies):.4f}")

# 2. Centralized training (pool all data - ideal but privacy-violating)
print("\n" + "="*60)
print("Training centralized model (privacy-violating baseline)...\n")

# Combine all training data
all_X_train = np.vstack([X for X, y in institutions_data])
all_y_train = np.concatenate([y for X, y in institutions_data])

centralized_model = LogisticRegression(max_iter=1000, random_state=42)
centralized_model.fit(all_X_train, all_y_train)
centralized_accuracy = centralized_model.score(global_test_X, global_test_y)

print(f"Centralized model accuracy: {centralized_accuracy:.4f}")

# 3. Federated learning (our approach)
federated_accuracy = fl_system.history['global_accuracy'][-1]

print("\n" + "="*60)
print("\n📊 COMPARISON SUMMARY\n")
print(f"{'Approach':<30} {'Accuracy':<15} {'Privacy-Preserving'}")
print("-" * 60)
print(f"{'Local-only (avg)':<30} {avg_local_accuracy:.4f}          {'✓'}")
print(f"{'Federated Learning':<30} {federated_accuracy:.4f}          {'✓'}")
print(f"{'Centralized (ideal)':<30} {centralized_accuracy:.4f}          {'✗'}")
print("="*60)

# Calculate improvement
improvement_vs_local = ((federated_accuracy - avg_local_accuracy) / avg_local_accuracy) * 100
gap_vs_centralized = centralized_accuracy - federated_accuracy

print(f"\n✨ Federated Learning Benefits:")
print(f"  • {improvement_vs_local:+.1f}% improvement over local-only training")
print(f"  • Only {gap_vs_centralized:.4f} accuracy gap vs centralized (privacy cost)")
print(f"  • Preserves patient privacy across all {len(institutions_data)} hospitals")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
approaches = ['Local-only\n(avg)', 'Federated\nLearning', 'Centralized\n(ideal)']
accuracies = [avg_local_accuracy, federated_accuracy, centralized_accuracy]
colors = ['coral', 'steelblue', 'lightgray']

bars = axes[0].bar(approaches, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0.6, 0.9])
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Privacy vs Performance trade-off
privacy_scores = [1.0, 1.0, 0.0]  # 1 = privacy-preserving, 0 = not

axes[1].scatter([privacy_scores[0]], [accuracies[0]], s=200, color=colors[0],
               alpha=0.7, edgecolors='black', linewidth=2, label='Local-only')
axes[1].scatter([privacy_scores[1]], [accuracies[1]], s=200, color=colors[1],
               alpha=0.7, edgecolors='black', linewidth=2, label='Federated')
axes[1].scatter([privacy_scores[2]], [accuracies[2]], s=200, color=colors[2],
               alpha=0.7, edgecolors='black', linewidth=2, label='Centralized')

axes[1].set_xlabel('Privacy Preservation', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Privacy-Performance Trade-off', fontsize=14, fontweight='bold')
axes[1].set_xlim([-0.2, 1.2])
axes[1].set_ylim([0.6, 0.9])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No Privacy', 'Full Privacy'])
axes[1].legend(loc='lower left')
axes[1].grid(alpha=0.3)

# Annotate the sweet spot
axes[1].annotate('Sweet Spot:\nHigh privacy + High accuracy',
                xy=(privacy_scores[1], accuracies[1]),
                xytext=(0.5, 0.65),
                arrowprops=dict(arrowstyle='->', lw=2, color='green'),
                fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 5: Privacy Analysis and Secure Aggregation

### Privacy Guarantees and Limitations

While federated learning prevents raw data sharing, it still has privacy vulnerabilities:

**Threats**:
1. **Gradient leakage**: Model updates can leak information about training data
2. **Model inversion**: Adversaries may reconstruct training samples from model weights
3. **Membership inference**: Attackers can determine if a specific patient was in training data

**Defense: Secure Aggregation**

Secure aggregation protocols (Bonawitz et al., 2017) ensure the server learns only the aggregated model, not individual updates:

$$
w_{\text{global}} = \sum_{k=1}^{K} w_k
$$

But the server cannot access any individual $w_k$.

**Implementation using Secret Sharing**:

1. Each institution $k$ splits its update $w_k$ into shares:
   $$w_k = s_{k,1} + s_{k,2} + ... + s_{k,K}$$

2. Institution $k$ sends share $s_{k,j}$ to institution $j$

3. Each institution sums received shares: $S_j = \sum_{k=1}^{K} s_{k,j}$

4. Server aggregates: $w_{\text{global}} = \sum_{j=1}^{K} S_j = \sum_{k=1}^{K} w_k$

**Combined with Differential Privacy**:

For strongest protection, combine FL with DP:
- Add noise to gradients before sharing (DP-SGD)
- Use secure aggregation to hide individual contributions
- Result: Formal privacy guarantee + protection against model inversion

In [ ]:
def simple_secure_aggregation_demo(local_updates, n_institutions):
    """
    Simplified demonstration of secure aggregation concept.

    In reality, this uses cryptographic protocols like:
    - Secret sharing (Shamir's scheme)
    - Masking with random values
    - Secure multi-party computation

    This demo shows the concept without full cryptographic implementation.
    """
    print("🔒 Secure Aggregation Protocol Demo\n")
    print("Simulating secure aggregation where server cannot see individual updates...\n")

    # Step 1: Each institution creates random masks
    print("Step 1: Institutions generate pairwise random masks")
    masks = {}
    for i in range(n_institutions):
        for j in range(i + 1, n_institutions):
            # Pairwise random mask (same for both institutions)
            mask = np.random.randn(*local_updates[0].shape) * 0.1
            masks[(i, j)] = mask

    # Step 2: Each institution masks its update
    print("Step 2: Institutions add/subtract masks to their updates")
    masked_updates = []
    for i in range(n_institutions):
        masked = local_updates[i].copy()
        # Add masks with institutions j > i, subtract masks with j < i
        for j in range(n_institutions):
            if i < j:
                masked += masks[(i, j)]
            elif i > j:
                masked -= masks[(j, i)]
        masked_updates.append(masked)

    print("Step 3: Server receives masked updates (individual updates are hidden)")
    print("Step 4: Server aggregates masked updates\n")

    # Step 3: Server aggregates (masks cancel out)
    secure_aggregate = sum(masked_updates) / n_institutions

    # Step 4: Compare with direct aggregation
    direct_aggregate = sum(local_updates) / n_institutions

    # Verify they're the same (masks cancelled)
    difference = np.linalg.norm(secure_aggregate - direct_aggregate)

    print("✅ Results:")
    print(f"  • Server computed correct aggregate: {difference < 1e-10}")
    print(f"  • Individual updates remain hidden from server")
    print(f"  • Difference (should be ~0): {difference:.2e}")

    return secure_aggregate

# Demo secure aggregation
print("=" * 70)
example_updates = [inst[0][0] for inst in institutions_data]  # First sample from each hospital
secure_result = simple_secure_aggregation_demo(example_updates, n_institutions=10)
print("=" * 70)

### Privacy-Performance Trade-off Analysis

In [ ]:
print("\n📋 Privacy and Security Considerations in Federated Learning:\n")

considerations = [
    {
        'aspect': 'Data Privacy',
        'protection': 'Raw patient data never leaves hospital',
        'limitation': 'Model updates may still leak some information'
    },
    {
        'aspect': 'Model Privacy',
        'protection': 'Secure aggregation hides individual contributions',
        'limitation': 'Global model is shared (potential model inversion)'
    },
    {
        'aspect': 'Communication',
        'protection': 'Only model weights transmitted (efficient)',
        'limitation': 'Multiple rounds needed (communication overhead)'
    },
    {
        'aspect': 'Differential Privacy',
        'protection': 'Can add DP noise to gradients (formal guarantee)',
        'limitation': 'Reduces model accuracy (privacy-utility trade-off)'
    },
    {
        'aspect': 'Compliance',
        'protection': 'Satisfies HIPAA/GDPR data minimization',
        'limitation': 'Requires institutional agreements and governance'
    }
]

for item in considerations:
    print(f"🔹 {item['aspect']}:")
    print(f"   ✓ Protection: {item['protection']}")
    print(f"   ⚠️  Limitation: {item['limitation']}")
    print()

print("="*70)
print("\n💡 Best Practices for Federated Learning in Healthcare:\n")

best_practices = [
    "Combine FL with differential privacy (DP-SGD) for formal privacy guarantees",
    "Use secure aggregation protocols to prevent server from seeing individual updates",
    "Implement secure enclaves (e.g., Intel SGX) for trusted execution",
    "Regularly audit for membership inference and model inversion attacks",
    "Establish clear data governance agreements between institutions",
    "Monitor for Byzantine attacks (malicious institutions sending bad updates)",
    "Use homomorphic encryption for operations on encrypted model updates",
    "Implement client selection strategies to ensure fair representation"
]

for i, practice in enumerate(best_practices, 1):
    print(f"{i}. {practice}")

---

## Part 6: Clinical Evaluation and Deployment

### Model Performance on Global Test Set

In [ ]:
# Detailed evaluation
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Get predictions
fl_predictions = fl_system.predict(global_test_X)
fl_probabilities = fl_system.predict_proba(global_test_X)

# Confusion matrix
cm = confusion_matrix(global_test_y, fl_predictions)
tn, fp, fn, tp = cm.ravel()

# Clinical metrics
sensitivity = tp / (tp + fn)  # Recall / True Positive Rate
specificity = tn / (tn + fp)  # True Negative Rate
ppv = tp / (tp + fp)  # Positive Predictive Value / Precision
npv = tn / (tn + fn)  # Negative Predictive Value

# ROC curve
fpr, tpr, thresholds = roc_curve(global_test_y, fl_probabilities)
roc_auc = auc(fpr, tpr)

print("🏥 Clinical Performance Metrics:\n")
print(f"Sensitivity (Recall):    {sensitivity:.4f}  ← Correctly identified {sensitivity:.1%} of patients with disorder")
print(f"Specificity:             {specificity:.4f}  ← Correctly identified {specificity:.1%} of healthy patients")
print(f"Positive Predictive Value: {ppv:.4f}  ← {ppv:.1%} of positive predictions are correct")
print(f"Negative Predictive Value: {npv:.4f}  ← {npv:.1%} of negative predictions are correct")
print(f"ROC AUC:                 {roc_auc:.4f}  ← Overall discriminative ability")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")

In [ ]:
# Visualize clinical performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['Predicted Negative', 'Predicted Positive'],
           yticklabels=['Actual Negative', 'Actual Positive'],
           cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# ROC curve
axes[1].plot(fpr, tpr, color='darkblue', lw=2,
            label=f'Federated Model (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Clinical Interpretation for Specific Patient

In [ ]:
# Example patient prediction
patient_idx = 42
patient_features = global_test_X[patient_idx]
true_label = global_test_y[patient_idx]
predicted_prob = fl_system.predict_proba(patient_features.reshape(1, -1))[0]
predicted_label = fl_system.predict(patient_features.reshape(1, -1))[0]

print("\n" + "="*70)
print("🏥 Clinical Decision Support: Patient Case Example")
print("="*70)
print(f"\nPatient ID: {patient_idx}")
print(f"\nModel Prediction:")
print(f"  Predicted Risk: {predicted_prob:.1%}")
print(f"  Classification: {'POSITIVE - Disorder likely present' if predicted_label == 1 else 'NEGATIVE - Disorder unlikely'}")
print(f"  Ground Truth: {'Positive' if true_label == 1 else 'Negative'}")
print(f"  Correct Prediction: {'✓ Yes' if predicted_label == true_label else '✗ No'}")

print(f"\n📋 Clinical Recommendation:")
if predicted_prob >= 0.7:
    print("  • HIGH RISK: Recommend genetic counseling and confirmatory testing")
    print("  • Consider specialist referral")
    print("  • Discuss family screening")
elif predicted_prob >= 0.3:
    print("  • MODERATE RISK: Additional clinical evaluation recommended")
    print("  • Schedule follow-up in 3-6 months")
    print("  • Monitor relevant biomarkers")
else:
    print("  • LOW RISK: Routine monitoring sufficient")
    print("  • Standard care pathway")
    print("  • Re-assess if symptoms develop")

print("\n⚠️  Important: This model was trained via federated learning across")
print("    10 hospitals WITHOUT sharing patient data, maintaining privacy")
print("    while leveraging collective knowledge.")
print("="*70)

---

## Summary and Key Takeaways

### What We Learned

1. **Federated Learning Fundamentals**:
   - FedAvg algorithm enables collaborative learning without data sharing
   - Model updates are aggregated using weighted averaging
   - Convergence achieved through multiple rounds of local training + global aggregation

2. **Performance Benefits**:
   - Federated learning significantly outperforms local-only training
   - Approaches centralized performance while preserving privacy
   - Particularly valuable for rare diseases with small per-hospital samples

3. **Privacy Protections**:
   - Raw patient data never leaves individual hospitals
   - Secure aggregation prevents server from seeing individual updates
   - Can be combined with differential privacy for formal guarantees

4. **Challenges**:
   - Non-IID data across institutions affects convergence
   - Communication overhead (multiple rounds required)
   - Potential for gradient leakage and model inversion attacks
   - Requires governance agreements and institutional coordination

### Clinical Applications

Federated learning is particularly well-suited for:

- **Rare disease prediction**: Pool limited cases across many centers
- **Multi-site clinical trials**: Analyze data without centralization
- **Regional health networks**: Collaborate while respecting local regulations
- **International collaborations**: Overcome cross-border data transfer restrictions
- **Pediatric medicine**: Leverage small sample sizes from children's hospitals

### Best Practices

1. **Technical**:
   - Combine FL with DP-SGD for stronger privacy guarantees
   - Use secure aggregation protocols (e.g., secure multi-party computation)
   - Implement Byzantine-robust aggregation to handle malicious participants
   - Monitor for convergence issues due to non-IID data

2. **Governance**:
   - Establish clear data use agreements between institutions
   - Define ownership of the collaborative model
   - Implement audit trails for model updates and access
   - Ensure compliance with local regulations (HIPAA, GDPR)

3. **Clinical**:
   - Validate federated models against centralized baselines when possible
   - Monitor for distribution shift across participating sites
   - Establish clinical oversight committees
   - Plan for model updates as new institutions join

### Limitations

- **Not a silver bullet**: FL still has privacy vulnerabilities (model inversion, membership inference)
- **Performance gap**: May not match centralized training, especially with extreme non-IID data
- **Communication costs**: Requires reliable networks and multiple rounds
- **Complexity**: More challenging to implement and maintain than centralized training

### Further Reading

- McMahan et al. (2017): "Communication-Efficient Learning of Deep Networks from Decentralized Data"
- Bonawitz et al. (2019): "Towards Federated Learning at Scale: System Design"
- Li et al. (2020): "Federated Learning on Non-IID Data Silos"
- Kaissis et al. (2020): "Secure, privacy-preserving and federated machine learning in medical imaging"
- Rieke et al. (2020): "The future of digital health with federated learning"

### Next Steps

In **Notebook 11.3**, we'll explore **Adversarial Robustness**, examining how to defend AI models against malicious attacks that attempt to manipulate predictions through carefully crafted inputs.

---

**End of Notebook 11.2**